# C3-gradient-descent — Practice p18 — Solution

The constant-rate floors fall from about $0.25624$ to $0.24414$ to $0.23669$ as $\eta$ shrinks from $0.1$ to $0.05$ to $0.01$. The tradeoff is speed: after 100 steps the $0.01$ run still has loss $0.44564$, much worse than the approximately $0.23858$ and $0.23683$ reached by the larger rates. The schedule matches the $0.1$ run's early loss exactly and ends at a floor about $0.23699$, close to the best constant-rate floor. Shrinking $\eta$ preserves fast early movement while reducing the scale of later noisy batch-gradient steps around the minimum.

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
X = rng.normal(0, 1, (200, 2))
w_true = np.array([1.0, -2.0])
b_true = 0.5
y = (X * w_true).sum(axis=1) + b_true + rng.normal(0, 0.5, 200)
batch_indices = rng.integers(0, 200, size=(2000, 5))

def mse_loss(w, b):
    return (((X * w).sum(axis=1) + b - y) ** 2).mean()

def run_sgd(constant_eta=None):
    w = np.zeros(2)
    b = 0.0
    losses = np.empty(2000)
    for step in range(2000):
        eta = constant_eta if constant_eta is not None else 0.1 * 0.5 ** (step // 500)
        idx = batch_indices[step]
        residuals = (X[idx] * w).sum(axis=1) + b - y[idx]
        grad_w = 2 * (residuals[:, None] * X[idx]).mean(axis=0)
        grad_b = 2 * residuals.mean()
        w = w - eta * grad_w
        b = b - eta * grad_b
        losses[step] = mse_loss(w, b)
    return losses

rates = np.array([0.1, 0.05, 0.01])
floors = np.empty(3)
at100 = np.empty(3)
for position, rate in enumerate(rates):
    losses = run_sgd(float(rate))
    floors[position] = losses[-200:].mean()
    at100[position] = losses[99]
losses_sched = run_sgd()
floor_sched = losses_sched[-200:].mean()
sched_at100 = losses_sched[99]
floors, at100, floor_sched, sched_at100

### Answer check

In [ ]:
assert np.allclose(floors, [0.256243213061, 0.244141730348, 0.236692931342], atol=1e-12, rtol=0)
assert np.allclose(at100, [0.238581048275, 0.236833204260, 0.445636843304], atol=1e-12, rtol=0)
assert np.isclose(floor_sched, 0.236993868117, atol=1e-12, rtol=0)
assert np.isclose(sched_at100, 0.238581048275, atol=1e-12, rtol=0)
assert np.isclose(sched_at100, at100[0], atol=1e-12, rtol=0)
assert abs(floor_sched - floors[2]) < 0.001